## 05 - Gold Layer Readiness
Cel: ocena czy tabele gold są gotowe pod dashboard — kompletność joinów, pokrycie między źródłami, gęstość danych.
Tabele: wszystkie gold.*

In [0]:
%sql
SELECT 'ohlcv_with_dimension' AS table_name, COUNT(*) AS rows FROM gold.ohlcv_with_dimension
UNION ALL
SELECT 'fred_with_dimension', COUNT(*) FROM gold.fred_with_dimension
UNION ALL
SELECT 'av_sentiment_aggregated', COUNT(*) FROM gold.av_sentiment_aggregated
UNION ALL
SELECT 'av_sentiment_sector_daily', COUNT(*) FROM gold.av_sentiment_sector_daily
UNION ALL
SELECT 'sentiment_vs_returns', COUNT(*) FROM gold.sentiment_vs_returns
UNION ALL
SELECT 'sentiment_lead_lag', COUNT(*) FROM gold.sentiment_lead_lag
UNION ALL
SELECT 'macro_impact_on_tech', COUNT(*) FROM gold.macro_impact_on_tech

In [0]:
%sql
SELECT 
  'ohlcv' AS source, MIN(date) AS min_date, MAX(date) AS max_date 
FROM silver.ohlcv_indicators
UNION ALL
SELECT 
  'sentiment', MIN(date), MAX(date) 
FROM gold.av_sentiment_aggregated
UNION ALL
SELECT 
  'fred', MIN(date), MAX(date) 
FROM silver.fred_macro_indicators

In [0]:
%sql
SELECT
  COUNT(DISTINCT o.symbol) AS ohlcv_symbols,
  COUNT(DISTINCT s.symbol) AS symbols_with_sentiment
FROM gold.ohlcv_with_dimension o
LEFT JOIN gold.av_sentiment_aggregated s
ON o.symbol = s.symbol
WHERE o.sector = 'technology'

In [0]:
%sql
SELECT 
  'Sentiment vs daily returns' AS dashboard_view,
  COUNT(*) AS data_points,
  COUNT(DISTINCT symbol) AS symbols
FROM gold.sentiment_vs_returns

UNION ALL

SELECT 
  'Sentiment lead/lag',
  COUNT(*),
  COUNT(DISTINCT symbol)
FROM gold.sentiment_lead_lag

UNION ALL

SELECT 
  'Sector sentiment trend',
  COUNT(*),
  COUNT(DISTINCT industry)
FROM gold.av_sentiment_sector_daily

UNION ALL

SELECT 
  'Macro impact on tech',
  COUNT(*),
  1
FROM gold.macro_impact_on_tech

### Wnioski

1. Gold layer: OHLCV 26845 i FRED 40634 to solidna baza. Sentiment 477-893 ograniczone. Macro impact 13 - minimalne.
2. Aktualny overlap trzech źródeł: marzec 2025 - marzec 2026, ograniczony przez OHLCV. Po rozszerzeniu do 2010 sytuacja znacząco się poprawi.
3. 100% pokrycie sentymentem dla tech symboli (41/41). Liczbowo pełne, jakościowo nierówne.
4. Dashboard readiness: sentiment vs returns i lead/lag gotowe (477-535 pts, 39-40 symboli). Sector sentiment gotowe z zastrzeżeniem filtrowania słabych industry. Macro impact tylko prosty widok - 13 punktów to za mało na analizę.
5. Wąskie gardła: OHLCV zakres dat (do rozszerzenia), sentyment nierówne pokrycie per symbol, macro impact zbyt mała próbka.